# Online Retail Sales Analysis and Customer Insights

**Data note:** bundled 2,000-row CSV is simulated practice data, not actual company transactions. See README for the authentic UCI source.

## 1. Load libraries and data

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'online_retail_2000_records.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['InvoiceDate'])
print('Rows:', len(df))
df.head()

## 2. Inspect and clean data

In [ ]:
print(df.info())
print('\nMissing values:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().dropna(subset=['InvoiceNo','Description','Quantity','UnitPrice','InvoiceDate','Country']).copy()
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
sales = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
sales['Month'] = sales['InvoiceDate'].dt.to_period('M').astype(str)
sales['Weekday'] = sales['InvoiceDate'].dt.day_name()
sales.describe(include='all')

## 3. Key performance indicators

In [ ]:
revenue = sales['TotalAmount'].sum()
invoices = sales['InvoiceNo'].nunique()
print(f'Total revenue: £{revenue:,.2f}')
print('Units sold:', int(sales['Quantity'].sum()))
print('Unique invoices:', invoices)
print('Unique customers:', sales['CustomerID'].nunique())
print(f'Average revenue per invoice: £{revenue/invoices:,.2f}' if invoices else 'No invoices')

## 4. Visualize sales

In [ ]:
monthly = sales.groupby('Month')['TotalAmount'].sum().sort_index()
monthly.plot(kind='line', marker='o', figsize=(10,4), title='Monthly Sales Revenue')
plt.ylabel('Revenue (GBP)'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

sales.groupby('Description')['Quantity'].sum().nlargest(10).sort_values().plot(kind='barh', figsize=(10,5), title='Top 10 Products by Quantity Sold')
plt.xlabel('Units sold'); plt.tight_layout(); plt.show()

sales.groupby('Country')['TotalAmount'].sum().nlargest(10).sort_values().plot(kind='barh', figsize=(10,5), title='Top Countries by Sales Revenue')
plt.xlabel('Revenue (GBP)'); plt.tight_layout(); plt.show()

## 5. Findings and limitations

Interpret outputs based on the data source. The bundled data is simulated for practice and must not be represented as verified real company data. For authentic analysis, download the UCI Online Retail dataset, select exactly 2,000 rows, and rerun the notebook.